# 📊 CAPM — Complete Data Science Course-Style Notebook
### Teacher: Aapka AI Data Science Instructor | Topic: Capital Asset Pricing Model
### Real Data: S&P 500 + Big Tech (`yfinance`) | Interactive Viz: `Plotly Express`

---

Salam Syed! 👋 Aaj hum CAPM ko **ek Data Science course ki tarah** step-by-step seekhenge. Har cell mein **thora sa code** hoga, aur uske upar/neeche **poori tarah samjhaya jayega** — jaise ek teacher class mein board par likh kar samjhata hai.

### Teaching Style Rules Jo Follow Karenge:
1. **Small cells** — har cell sirf **1 concept** ya **1-3 lines** ka kaam karega (bade blocks nahi)
2. Har naye pandas/numpy function ka pehli baar use hone par **mini-explanation**
3. **Plotly Express** se interactive charts (jinko aap zoom/hover kar sakte ho)
4. Real pandas concepts jaise `.drop()`, `inplace=True`, `.loc` vs `.iloc`, `.dropna()`, `.fillna()` — sab explicitly demonstrate honge

### Course Outline:
1. Environment Setup
2. Data Acquisition (yfinance)
3. **Pandas Fundamentals Detour** (drop, inplace, loc/iloc, dropna, fillna)
4. Exploratory Data Analysis (EDA) with Plotly
5. Returns Engineering
6. Statistical Foundations
7. Beta Estimation (3 Methods)
8. Rolling Beta Analysis
9. CAPM Model Application
10. Security Market Line (Interactive)
11. Alpha & Performance Attribution
12. Risk-Adjusted Metrics
13. Portfolio-Level CAPM
14. Final Risk-Return Dashboard
15. Limitations, Conclusion & Further Study


---
## 📦 PART 1: Environment Setup

### Cell 1.1 — Core Numerical & Data Libraries

In [1]:
import numpy as np    # Numerical computing: arrays, math, linear algebra
import pandas as pd   # DataFrames: tabular data handling (jaise Excel, lekin code se)

C:\Users\SMIT\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\SMIT\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


**Teacher's Note:** `numpy` hume fast array operations deta hai (Python ki normal lists se kaafi tez). `pandas` hume **DataFrame** deta hai — ek table jaisa structure jisme rows = dates, columns = stocks hongi.

In [2]:
import matplotlib.pyplot as plt   # Static plotting (basic charts)
import matplotlib.dates as mdates  # Date formatting on x-axis
import seaborn as sns              # Statistical charts (heatmaps waghera), matplotlib ke upar

C:\Users\Administrator\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
import plotly.express as px        # Interactive charts — ek line mein professional graphs
import plotly.graph_objects as go  # Zyada control ke liye — jab px kaafi na ho
import plotly.io as pio            # Renderer settings (kaise chart display ho)

pio.renderers.default = "notebook"  # Notebook ke andar hi interactive chart render hoga

**Plotly kyun?** `matplotlib` static images banata hai (zoom/hover nahi kar sakte). `plotly.express` (aliased as `px`) sirf **1-2 lines** mein aisa chart banata hai jisme aap **hover kar ke exact values** dekh sakte ho, zoom kar sakte ho, legend par click kar ke series hide kar sakte ho. Data Science mein `px` industry-standard hai quick EDA ke liye.

In [4]:
from scipy import stats           # Statistical functions (yahan: linear regression)
import statsmodels.api as sm      # Advanced regression with full statistical summary

In [5]:
try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except ImportError:
    YFINANCE_AVAILABLE = False

print("yfinance available:", YFINANCE_AVAILABLE)

yfinance available: True


### Cell 1.2 — Reproducibility & Style Settings

In [68]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

sns.set_theme(style="darkgrid", palette="husl")

plt.rcParams["figure.figsize"] = (10, 5.5)

print("✅ Environment Ready")

✅ Environment Ready


In [70]:
plt.style.use("ggplot")

In [74]:
import matplotlib
import seaborn

print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", seaborn.__version__)

Matplotlib: 3.5.2
Seaborn: 0.11.2


In [76]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Is line ko use mat karo
# plt.style.use("seaborn-v0_8-darkgrid")

sns.set_theme(style="darkgrid", palette="husl")

plt.rcParams["figure.figsize"] = (10, 5.5)

print("✅ Environment Ready")

✅ Environment Ready


---
## 📥 PART 2: Data Acquisition

### Cell 2.1 — Ticker Symbols Define Karna

**Teacher's Note:** Har company/index ka stock market mein ek unique code hota hai jise **ticker** kehte hain. `^GSPC` S&P 500 index ka ticker hai (jise hum "market" ka proxy maanenge).

In [7]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META']
MARKET_TICKER = '^GSPC'
ALL_TICKERS = [MARKET_TICKER] + TICKERS

print("Companies:", TICKERS)
print("Market Proxy:", MARKET_TICKER)

Companies: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META']
Market Proxy: ^GSPC


### Cell 2.2 — Date Range Define Karna

In [8]:
START_DATE = '2023-01-01'
END_DATE = '2025-01-01'

print(f"Data range: {START_DATE} to {END_DATE}")

Data range: 2023-01-01 to 2025-01-01


### Cell 2.3 — Fallback Data Generator (Offline Safety Net)

**Teacher's Note:** Kabhi kabhi internet/API access na ho (jaise sandboxed environments). Isliye ek **fallback function** likhte hain jo realistic simulated price data banata hai — taake notebook **kabhi crash na ho**.

In [9]:
def generate_synthetic_data(tickers, market_ticker, start, end, seed=42):
    '''Realistic simulated OHLC price data - offline fallback.'''
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start=start, end=end)
    n = len(dates)
    market_daily = rng.normal(0.00045, 0.010, n)
    market_price = 4000 * (1 + market_daily).cumprod()
    return dates, n, market_daily, market_price

In [10]:
approx_betas = {'AAPL': 1.25, 'MSFT': 1.05, 'GOOGL': 1.10, 'AMZN': 1.30, 'NVDA': 1.75, 'META': 1.30}
start_prices = {'AAPL': 130, 'MSFT': 240, 'GOOGL': 90, 'AMZN': 95, 'NVDA': 15, 'META': 120}

print("Approximate real-world betas (public ballpark figures):")
for k, v in approx_betas.items():
    print(f"  {k}: β ≈ {v}")

Approximate real-world betas (public ballpark figures):
  AAPL: β ≈ 1.25
  MSFT: β ≈ 1.05
  GOOGL: β ≈ 1.1
  AMZN: β ≈ 1.3
  NVDA: β ≈ 1.75
  META: β ≈ 1.3


In [11]:
def build_synthetic_prices(tickers, market_ticker, start, end, seed=42):
    dates, n, market_daily, market_price = generate_synthetic_data(tickers, market_ticker, start, end, seed)
    rng = np.random.default_rng(seed + 1)
    price_data = {market_ticker: market_price}
    for t in tickers:
        beta = approx_betas.get(t, 1.0)
        idio_noise = rng.normal(0.0003, 0.014, n)
        stock_daily = beta * market_daily + idio_noise
        price_data[t] = start_prices.get(t, 100) * (1 + stock_daily).cumprod()
    df_prices = pd.DataFrame(price_data, index=dates)
    df_prices.index.name = 'Date'
    return df_prices

### Cell 2.4 — Real Data Download Attempt (yfinance)

In [12]:
if YFINANCE_AVAILABLE:
    try:
        raw = yf.download(ALL_TICKERS, start=START_DATE, end=END_DATE, progress=False)
        prices = raw['Close'] if 'Close' in raw.columns.get_level_values(0) else raw['Adj Close']
        prices = prices.dropna(how='all')
        if prices.empty or prices.isna().all().all():
            raise ValueError("No data returned - likely no internet access")
        DATA_SOURCE = "yfinance (LIVE Yahoo Finance data)"
    except Exception as e:
        print(f"⚠️ Live download failed: {e}")
        prices = None
else:
    prices = None
    print("yfinance not available")

### Cell 2.5 — Fallback Trigger (Agar Live Data Na Mile)

In [13]:
if prices is None or prices.empty:
    prices = build_synthetic_prices(TICKERS, MARKET_TICKER, START_DATE, END_DATE)
    DATA_SOURCE = "Simulated (offline fallback, realistic approximate betas)"

print("Data Source Used:", DATA_SOURCE)

Data Source Used: yfinance (LIVE Yahoo Finance data)


### Cell 2.6 — Column Rename (Market Ticker → Readable Name)

In [14]:
prices = prices.rename(columns={MARKET_TICKER: 'Market_SP500'})
prices.columns

Index(['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'Market_SP500'], dtype='object', name='Ticker')

---
## 🧹 PART 3: Pandas Fundamentals Detour (Data Cleaning Concepts)

**Teacher's Note:** Isse pehle ke hum aage badhein, chalo pandas ke kuch **core concepts** seekh lete hain jo har Data Scientist ko aani chahiye: `.head()`, `.info()`, `dropna()`, `fillna()`, `.drop()`, `inplace=True`, aur `.loc`/`.iloc`.

### Cell 3.1 — `.head()` aur `.tail()`: Pehli/Aakhri Rows Dekhna

In [15]:
prices.head()   # Pehli 5 rows dikhata hai

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Market_SP500
Date,,,,,,,
2023-01-03,122.982704,85.820000,88.336700,123.654121,232.948257,14.283261,3824.139893
2023-01-04,124.251183,85.139999,87.305840,126.261230,222.758362,14.716301,3852.969971
2023-01-05,122.933548,83.120003,85.442360,125.834984,216.156326,14.233376,3808.100098
2023-01-06,127.456772,86.080002,86.572342,128.888168,218.703781,14.826055,3895.080078
2023-01-09,127.977936,87.360001,87.246353,128.342941,220.833160,15.593352,3892.090088


In [16]:
prices.tail(3)   # Aakhri 3 rows (number specify kar sakte ho)

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Market_SP500
Date,,,,,,,
2024-12-27,253.967392,223.750000,191.758408,596.859863,425.482513,136.805649,5970.839844
2024-12-30,250.598923,221.300003,190.246307,588.331970,419.849396,137.284958,5906.939941
2024-12-31,248.830215,219.389999,188.316360,582.630188,416.558380,134.089737,5881.629883


### Cell 3.2 — `.shape` aur `.columns`: Data Ka Size Aur Naam

In [17]:
print("Shape (rows, columns):", prices.shape)
print("Columns:", list(prices.columns))

Shape (rows, columns): (502, 7)
Columns: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'Market_SP500']


### Cell 3.3 — `.info()`: Data Types Aur Non-Null Counts

In [18]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 502 entries, 2023-01-03 to 2024-12-31
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   AAPL          502 non-null    float64
 1   AMZN          502 non-null    float64
 2   GOOGL         502 non-null    float64
 3   META          502 non-null    float64
 4   MSFT          502 non-null    float64
 5   NVDA          502 non-null    float64
 6   Market_SP500  502 non-null    float64
dtypes: float64(7)
memory usage: 31.4 KB


### Cell 3.4 — Missing Values Check Karna

**Teacher's Note:** `.isna()` har cell ko `True`/`False` mein convert karta hai (missing hai ya nahi). `.sum()` lagane se column-wise **total missing count** milta hai.

In [19]:
prices.isna().sum()

Ticker
AAPL            0
AMZN            0
GOOGL           0
META            0
MSFT            0
NVDA            0
Market_SP500    0
dtype: int64

### Cell 3.5 — `.fillna()`: Missing Values Bharna

`method='ffill'` (forward-fill) ka matlab hai: agar aaj ki price missing hai, to **pichle available din** ki price copy kar do. Ye stock data ke liye standard practice hai (weekends/holidays ke liye).

In [20]:
prices_filled = prices.ffill()   # ffill() = forward-fill (naye pandas mein fillna(method='ffill') deprecated hai)
prices_filled.isna().sum()

Ticker
AAPL            0
AMZN            0
GOOGL           0
META            0
MSFT            0
NVDA            0
Market_SP500    0
dtype: int64

### Cell 3.6 — `dropna()`: Baaki Bachi Missing Rows Hataana

Agar shuru ki rows mein bhi NaN ho (jaise pehla din), to `dropna()` un rows ko **completely remove** kar deta hai.

In [21]:
prices_clean = prices_filled.dropna()
print(f"Before dropna: {len(prices_filled)} rows | After dropna: {len(prices_clean)} rows")

Before dropna: 502 rows | After dropna: 502 rows


### Cell 3.7 — `.drop()`: Ek Column Ko Temporarily Hataana (Demo)

Kabhi kabhi hume ek column **remove** karna hota hai (permanently ya temporarily). `.drop(columns=[...])` se ye hota hai. Yahan hum sirf **demo** ke liye `NVDA` column ko drop kar rahe hain ek naye variable mein — original data safe rehta hai.

In [22]:
demo_without_nvda = prices_clean.drop(columns=['NVDA'])
print("Original columns:", list(prices_clean.columns))
print("After drop (demo only):", list(demo_without_nvda.columns))

Original columns: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'Market_SP500']
After drop (demo only): ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'Market_SP500']


### Cell 3.8 — `inplace=True` Ka Matlab

Normally `.drop()` ek **naya DataFrame return** karta hai, original change nahi hota (jaisa upar dekha). Agar `inplace=True` pass karein, to original DataFrame **khud modify** ho jata hai, aur function kuch return nahi karta (`None`).

⚠️ **Important:** `inplace=True` professional pipelines mein kam use hota hai kyunke debug karna mushkil ho jata hai. Best practice: `df = df.drop(...)` likhna (jaisa humne upar kiya).

In [23]:
demo_copy = prices_clean.copy()          # Pehle ek copy banate hain taake original safe rahe
demo_copy.drop(columns=['META'], inplace=True)   # inplace=True: demo_copy khud modify ho gaya
print("demo_copy columns after inplace drop:", list(demo_copy.columns))
print("Original prices_clean unaffected:", list(prices_clean.columns))

demo_copy columns after inplace drop: ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'NVDA', 'Market_SP500']
Original prices_clean unaffected: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'Market_SP500']


### Cell 3.9 — `.loc` vs `.iloc`: Label vs Position Se Data Select Karna

- **`.loc[label]`** → naam/date **label** se select karta hai
- **`.iloc[position]`** → row/column ka **numeric position** (0, 1, 2...) se select karta hai

In [24]:
first_date = prices_clean.index[0]
print("Using .loc (by date label):")
print(prices_clean.loc[first_date])

Using .loc (by date label):
Ticker
AAPL             122.982704
AMZN              85.820000
GOOGL             88.336700
META             123.654121
MSFT             232.948257
NVDA              14.283261
Market_SP500    3824.139893
Name: 2023-01-03 00:00:00, dtype: float64


In [25]:
print("Using .iloc (by numeric position, row 0):")
print(prices_clean.iloc[0])

Using .iloc (by numeric position, row 0):
Ticker
AAPL             122.982704
AMZN              85.820000
GOOGL             88.336700
META             123.654121
MSFT             232.948257
NVDA              14.283261
Market_SP500    3824.139893
Name: 2023-01-03 00:00:00, dtype: float64


### Cell 3.10 — Final Cleaned Dataset Confirm Karna

In [26]:
prices = prices_clean   # Aage se hum yahi clean version use karenge
print(f"📅 Date Range: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"📊 Total Trading Days: {len(prices)}")
print(f"📡 Data Source: {DATA_SOURCE}")

📅 Date Range: 2023-01-03 to 2024-12-31
📊 Total Trading Days: 502
📡 Data Source: yfinance (LIVE Yahoo Finance data)


---
## 📊 PART 4: Exploratory Data Analysis (EDA) — With Plotly Express

### Cell 4.1 — Summary Statistics

In [27]:
prices.describe().round(2)

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Market_SP500
count,502.00,502.00,502.00,502.00,502.00,502.00,502.00
mean,187.90,153.13,140.25,382.95,360.52,72.22,4858.26
std,28.06,36.50,27.37,138.27,62.11,41.09,649.88
min,122.93,83.12,85.44,123.65,216.16,14.23,3808.10
25%,169.79,127.03,122.41,284.31,317.90,41.88,4295.16
50%,183.14,153.25,137.26,351.37,370.45,49.71,4760.02
75%,212.44,183.48,163.64,501.40,412.69,115.26,5435.94
max,257.38,232.93,195.64,629.06,460.33,148.65,6090.27


### Cell 4.2 — Normalize Prices (Base = 100)

**Teacher's Note:** NVDA ~$15 hai aur MSFT ~$240 — direct compare karna misleading hoga. Isliye hum sab ko **same starting point (100)** par le aate hain, taake **% growth** compare ho sake.

In [28]:
normalized = (prices / prices.iloc[0]) * 100
normalized.head()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Market_SP500
Date,,,,,,,
2023-01-03,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
2023-01-04,101.031428,99.207644,98.833032,102.108388,95.625683,103.031798,100.753897
2023-01-05,99.960030,96.853884,96.723513,101.763680,92.791562,99.650740,99.580565
2023-01-06,103.637965,100.302962,98.002689,104.232812,93.885133,103.800205,101.855063
2023-01-09,104.061735,101.794455,98.765692,103.791883,94.799233,109.172212,101.776875


### Cell 4.3 — Interactive Price Chart (Plotly Express `px.line`)

**Teacher's Note:** `px.line()` ek interactive line chart banata hai. `x` aur `y` specify karne ke bajaye, hum poora DataFrame de dete hain — Plotly automatically har column ko ek line bana deta hai.

In [30]:
fig1 = px.line(
    normalized,
    x=normalized.index,
    y=normalized.columns,
    title="📈 Normalized Price Comparison (Base = 100) — Interactive",
    labels={'value': 'Normalized Price', 'Date': 'Date', 'variable': 'Ticker'}
)
fig1.update_layout(hovermode='x unified', legend_title_text='Ticker')
fig1.show()

C:\Users\Administrator\anaconda3\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



**Try This:** Chart par mouse le jao — exact values dikhengi. Legend mein kisi ticker par click karo — wo line hide/show hogi!

### Cell 4.4 — Daily Returns Calculate Karna (`.pct_change()`)

$$ R_t = \frac{P_t - P_{t-1}}{P_{t-1}} $$

In [31]:
simple_returns = prices.pct_change().dropna()
simple_returns.head()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Market_SP500
Date,,,,,,,
2023-01-04,0.010314,-0.007924,-0.011670,0.021084,-0.043743,0.030318,0.007539
2023-01-05,-0.010605,-0.023726,-0.021344,-0.003376,-0.029638,-0.032816,-0.011646
2023-01-06,0.036794,0.035611,0.013225,0.024263,0.011785,0.041640,0.022841
2023-01-09,0.004089,0.014870,0.007786,-0.004230,0.009736,0.051753,-0.000768
2023-01-10,0.004456,0.028732,0.004545,0.027188,0.007617,0.017981,0.006978


### Cell 4.5 — Log Returns (Alternative Method)

$$ r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) $$

Log returns **time-additive** hote hain — professional quant finance mein prefer kiye jate hain.

In [32]:
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.head()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Market_SP500
Date,,,,,,,
2023-01-04,0.010261,-0.007955,-0.011738,0.020865,-0.044729,0.029867,0.007511
2023-01-05,-0.010661,-0.024012,-0.021575,-0.003382,-0.030086,-0.033366,-0.011714
2023-01-06,0.036133,0.034992,0.013138,0.023974,0.011716,0.040796,0.022584
2023-01-09,0.004081,0.014760,0.007755,-0.004239,0.009689,0.050459,-0.000768
2023-01-10,0.004446,0.028327,0.004534,0.026825,0.007588,0.017821,0.006954


### Cell 4.6 — Returns Distribution (Interactive Box Plot with `px.box`)

In [33]:
fig2 = px.box(
    simple_returns * 100,
    title="📦 Daily Returns Distribution — Box Plot (Interactive)",
    labels={'value': 'Daily Return (%)', 'variable': 'Ticker'},
    points='outliers'
)
fig2.show()

**Teacher's Note:** Box plot mein box ka beech ka line = median, box ki height = middle 50% data ka spread, aur dots = outliers (extreme days).

### Cell 4.7 — Returns Histogram (Interactive with `px.histogram`)

In [34]:
fig3 = px.histogram(
    simple_returns['AAPL'] * 100,
    nbins=50,
    title="📊 AAPL Daily Returns Distribution (Histogram)",
    labels={'value': 'Daily Return (%)'}
)
fig3.update_layout(showlegend=False)
fig3.show()

---
## 📐 PART 5: Statistical Foundations

### Cell 5.1 — Annualization Constants

In [35]:
TRADING_DAYS = 252   # Ek saal mein approx trading days
print(f"Trading days per year assumption: {TRADING_DAYS}")

Trading days per year assumption: 252


### Cell 5.2 — Annualized Return Calculate Karna

$$ \text{Annual Return} = \bar{R}_{daily} \times 252 $$

In [36]:
annual_return = simple_returns.mean() * TRADING_DAYS
annual_return.round(4)

Ticker
AAPL            0.3774
AMZN            0.5193
GOOGL           0.4236
META            0.8504
MSFT            0.3182
NVDA            1.2528
Market_SP500    0.2249
dtype: float64

### Cell 5.3 — Annualized Volatility Calculate Karna

$$ \text{Annual Volatility} = \sigma_{daily} \times \sqrt{252} $$

In [37]:
annual_volatility = simple_returns.std() * np.sqrt(TRADING_DAYS)
annual_volatility.round(4)

Ticker
AAPL            0.2135
AMZN            0.3062
GOOGL           0.2921
META            0.3819
MSFT            0.2268
NVDA            0.5049
Market_SP500    0.1287
dtype: float64

### Cell 5.4 — Sab Kuch Ek DataFrame Mein Combine Karna

In [38]:
annual_stats = pd.DataFrame({
    'Annualized Return (%)': (annual_return * 100).round(2),
    'Annualized Volatility (%)': (annual_volatility * 100).round(2)
}).sort_values('Annualized Return (%)', ascending=False)

annual_stats

,Annualized Return (%),Annualized Volatility (%)
Ticker,,
NVDA,125.28,50.49
META,85.04,38.19
AMZN,51.93,30.62
GOOGL,42.36,29.21
AAPL,37.74,21.35
MSFT,31.82,22.68
Market_SP500,22.49,12.87


### Cell 5.5 — Correlation Matrix (`.corr()`)

In [39]:
corr_matrix = simple_returns.corr()
corr_matrix.round(2)

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Market_SP500
Ticker,,,,,,,
AAPL,1.00,0.39,0.44,0.38,0.51,0.34,0.62
AMZN,0.39,1.00,0.57,0.58,0.62,0.41,0.65
GOOGL,0.44,0.57,1.00,0.51,0.54,0.36,0.58
META,0.38,0.58,0.51,1.00,0.55,0.40,0.56
MSFT,0.51,0.62,0.54,0.55,1.00,0.50,0.67
NVDA,0.34,0.41,0.36,0.40,0.50,1.00,0.60
Market_SP500,0.62,0.65,0.58,0.56,0.67,0.60,1.00


### Cell 5.6 — Interactive Correlation Heatmap (`px.imshow`)

In [40]:
fig4 = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title="🔥 Correlation Heatmap — Daily Returns (Interactive)"
)
fig4.show()

---
## 📉 PART 6: Beta (β) Estimation — 3 Professional Methods

### Cell 6.1 — Market Returns Ko Alag Variable Mein Nikalna

In [41]:
market_ret = simple_returns['Market_SP500']
market_ret.head()

Date
2023-01-04    0.007539
2023-01-05   -0.011646
2023-01-06    0.022841
2023-01-09   -0.000768
2023-01-10    0.006978
Name: Market_SP500, dtype: float64

### Cell 6.2 — Method 1: Covariance / Variance Formula

$$ \beta_i = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)} $$

In [42]:
def beta_covariance(stock_ret, market_ret):
    cov_matrix = np.cov(stock_ret, market_ret)
    return cov_matrix[0, 1] / cov_matrix[1, 1]

beta_covariance(simple_returns['AAPL'], market_ret)

1.0339973601067811

### Cell 6.3 — Method 2: Simple Linear Regression (`scipy.stats.linregress`)

In [43]:
def beta_scipy_regression(stock_ret, market_ret):
    slope, intercept, r_value, p_value, std_err = stats.linregress(market_ret, stock_ret)
    return slope, intercept, r_value**2, p_value

beta_scipy_regression(simple_returns['AAPL'], market_ret)

(1.033997360106781,
 0.0005748257946902944,
 0.3882100401477756,
 3.266393086869591e-55)

### Cell 6.4 — Sab Tickers Ke Liye Beta Calculate Karna (Loop)

In [44]:
beta_results = []
for ticker in TICKERS:
    stock_ret = simple_returns[ticker]
    b_cov = beta_covariance(stock_ret, market_ret)
    b_reg, alpha_reg, r_sq, p_val = beta_scipy_regression(stock_ret, market_ret)
    beta_results.append({
        'Ticker': ticker,
        'Beta (Covariance)': round(b_cov, 3),
        'Beta (Regression)': round(b_reg, 3),
        'Alpha (Daily)': round(alpha_reg, 5),
        'R-Squared': round(r_sq, 3),
        'P-Value': f"{p_val:.2e}"
    })

beta_df = pd.DataFrame(beta_results)
beta_df

,Ticker,Beta (Covariance),Beta (Regression),Alpha (Daily),R-Squared,P-Value
0,AAPL,1.034,1.034,0.00057,0.388,3.27e-55
1,MSFT,1.183,1.183,0.00021,0.451,6.38e-67
2,GOOGL,1.314,1.314,0.00051,0.335,3.69e-46
3,AMZN,1.540,1.540,0.00069,0.419,9.03e-61
4,NVDA,2.344,2.344,0.00288,0.357,8.86e-50
5,META,1.667,1.667,0.00189,0.315,5.60e-43


### Cell 6.5 — Interactive Scatter + Regression Line (`px.scatter` with `trendline='ols'`)

**Teacher's Note:** Plotly Express ek magical parameter deta hai: `trendline='ols'` — ye khud regression line fit kar deta hai, humein manually calculate nahi karna padta plotting ke liye!

In [45]:
fig5 = px.scatter(
    simple_returns, x='Market_SP500', y='AAPL',
    trendline='ols',
    title="AAPL vs Market Returns — Beta = Regression Slope",
    labels={'Market_SP500': 'Market (S&P 500) Daily Return', 'AAPL': 'AAPL Daily Return'}
)
fig5.show()

### Cell 6.6 — Method 3: `statsmodels` OLS — Full Statistical Summary

In [46]:
X = sm.add_constant(simple_returns['Market_SP500'])   # Independent variable + intercept
y = simple_returns['AAPL']                              # Dependent variable

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                   AAPL   R-squared:                       0.388
Model:                            OLS   Adj. R-squared:                  0.387
Method:                 Least Squares   F-statistic:                     316.6
Date:                Sun, 12 Jul 2026   Prob (F-statistic):           3.27e-55
Time:                        17:51:31   Log-Likelihood:                 1571.3
No. Observations:                 501   AIC:                            -3139.
Df Residuals:                     499   BIC:                            -3130.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0006      0.000      1.214   

**Teacher's Note — Summary Kaise Padhein:**
- `const` row → Alpha (intercept)
- `Market_SP500` row → **ye Beta hai**
- `P>|t|` → 0.05 se kam ho to result statistically significant hai
- `R-squared` → market kitna % variation explain karta hai stock returns mein

---
## 🔄 PART 7: Rolling Beta Analysis

### Cell 7.1 — Rolling Window Size Define Karna

In [47]:
ROLLING_WINDOW = 60   # 60 trading days (~3 months)
print(f"Rolling window: {ROLLING_WINDOW} days")

Rolling window: 60 days


### Cell 7.2 — Rolling Covariance Aur Variance

In [48]:
rolling_cov = simple_returns[TICKERS].rolling(ROLLING_WINDOW).cov(simple_returns['Market_SP500'])
rolling_var = simple_returns['Market_SP500'].rolling(ROLLING_WINDOW).var()
rolling_cov.tail(3)

Ticker,AAPL,MSFT,GOOGL,AMZN,NVDA,META
Date,,,,,,
2024-12-27,0.000042,0.000074,0.000075,0.000106,0.000100,0.000078
2024-12-30,0.000045,0.000077,0.000077,0.000108,0.000101,0.000082
2024-12-31,0.000045,0.000078,0.000077,0.000106,0.000101,0.000080


### Cell 7.3 — Rolling Beta Formula Apply Karna

In [49]:
rolling_betas = rolling_cov.div(rolling_var, axis=0).dropna()
rolling_betas.tail(3)

Ticker,AAPL,MSFT,GOOGL,AMZN,NVDA,META
Date,,,,,,
2024-12-27,0.669407,1.177905,1.193035,1.681358,1.591330,1.246189
2024-12-30,0.688783,1.179490,1.183998,1.658008,1.555871,1.260599
2024-12-31,0.701751,1.208449,1.204317,1.645406,1.573959,1.238998


### Cell 7.4 — Interactive Rolling Beta Chart

In [50]:
fig6 = px.line(
    rolling_betas, x=rolling_betas.index, y=rolling_betas.columns,
    title=f"📉 Rolling {ROLLING_WINDOW}-Day Beta Over Time (Interactive)",
    labels={'value': 'Beta (β)', 'Date': 'Date', 'variable': 'Ticker'}
)
fig6.add_hline(y=1.0, line_dash='dash', line_color='black', annotation_text='Market Beta = 1.0')
fig6.update_layout(hovermode='x unified')
fig6.show()

C:\Users\Administrator\anaconda3\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



**Insight:** Agar line consistently upar-neeche move ho rahi hai, matlab company ka risk profile time ke sath badal raha hai — CAPM ka static-beta assumption yahan weak pad jata hai.

---
## 💰 PART 8: CAPM Model Application

### Cell 8.1 — Risk-Free Rate Assumption

In [51]:
RISK_FREE_RATE = 0.045   # ~4.5% (approx US 10-Year Treasury Yield)
print(f"Risk-Free Rate: {RISK_FREE_RATE*100:.2f}%")

Risk-Free Rate: 4.50%


### Cell 8.2 — Expected Market Return (Actual S&P 500 Annualized Return)

In [52]:
expected_market_return = annual_stats.loc['Market_SP500', 'Annualized Return (%)'] / 100
print(f"Expected Market Return: {expected_market_return*100:.2f}%")

Expected Market Return: 22.49%


### Cell 8.3 — Market Risk Premium

$$ \text{Market Risk Premium} = E(R_m) - R_f $$

In [53]:
market_risk_premium = expected_market_return - RISK_FREE_RATE
print(f"Market Risk Premium: {market_risk_premium*100:.2f}%")

Market Risk Premium: 17.99%


### Cell 8.4 — CAPM Formula Apply Karna (Loop)

$$ E(R_i) = R_f + \beta_i (E(R_m) - R_f) $$

In [54]:
capm_results = []
for _, row in beta_df.iterrows():
    ticker = row['Ticker']
    beta = row['Beta (Regression)']
    capm_expected = RISK_FREE_RATE + beta * market_risk_premium
    actual_annual = annual_stats.loc[ticker, 'Annualized Return (%)'] / 100
    capm_results.append({
        'Ticker': ticker,
        'Beta': beta,
        'CAPM Expected Return (%)': round(capm_expected * 100, 2),
        'Actual Annualized Return (%)': round(actual_annual * 100, 2),
        "Jensen's Alpha (%)": round((actual_annual - capm_expected) * 100, 2)
    })

capm_df = pd.DataFrame(capm_results).sort_values('Beta', ascending=False)
capm_df

,Ticker,Beta,CAPM Expected Return (%),Actual Annualized Return (%),Jensen's Alpha (%)
4,NVDA,2.344,46.67,125.28,78.61
5,META,1.667,34.49,85.04,50.55
3,AMZN,1.540,32.20,51.93,19.73
2,GOOGL,1.314,28.14,42.36,14.22
1,MSFT,1.183,25.78,31.82,6.04
0,AAPL,1.034,23.10,37.74,14.64


### Cell 8.5 — Interactive Bar Chart: Beta Comparison

In [55]:
fig7 = px.bar(
    capm_df.sort_values('Beta'), x='Beta', y='Ticker', orientation='h',
    color='Beta', color_continuous_scale='Viridis',
    title="📊 Beta Comparison Across Companies (Interactive)",
    text='Beta'
)
fig7.add_vline(x=1.0, line_dash='dash', line_color='red', annotation_text='Market Beta = 1.0')
fig7.show()

---
## 📏 PART 9: Security Market Line (SML) — Interactive

### Cell 9.1 — SML Line Ke Points Generate Karna

In [56]:
beta_range = np.linspace(0, 2.2, 100)
sml_returns = (RISK_FREE_RATE + beta_range * market_risk_premium) * 100
sml_df = pd.DataFrame({'Beta': beta_range, 'Expected Return (%)': sml_returns})
sml_df.head()

,Beta,Expected Return (%)
0,0.000000,4.500000
1,0.022222,4.899778
2,0.044444,5.299556
3,0.066667,5.699333
4,0.088889,6.099111


### Cell 9.2 — Interactive SML Plot (`go.Figure` — Line + Scatter Combined)

In [57]:
fig8 = go.Figure()

fig8.add_trace(go.Scatter(
    x=sml_df['Beta'], y=sml_df['Expected Return (%)'],
    mode='lines', name='Security Market Line (SML)',
    line=dict(color='black', width=3)
))

fig8.add_trace(go.Scatter(
    x=capm_df['Beta'], y=capm_df['Actual Annualized Return (%)'],
    mode='markers+text', name='Companies',
    text=capm_df['Ticker'], textposition='top center',
    marker=dict(size=14, color=capm_df['Beta'], colorscale='Viridis', showscale=True, line=dict(width=1, color='black'))
))

fig8.add_trace(go.Scatter(
    x=[1], y=[expected_market_return*100], mode='markers', name='Market (S&P 500)',
    marker=dict(size=16, symbol='diamond', color='purple')
))

fig8.add_trace(go.Scatter(
    x=[0], y=[RISK_FREE_RATE*100], mode='markers', name='Risk-Free Asset',
    marker=dict(size=16, symbol='star', color='red')
))

fig8.update_layout(
    title="📐 Security Market Line — Interactive (Hover for details)",
    xaxis_title="Beta (β) — Systematic Risk",
    yaxis_title="Annualized Return (%)",
    hovermode='closest'
)
fig8.show()

**Teacher's Note:** Line ke **upar** points = undervalued (market se zyada return de rahe hain us risk ke liye). Line ke **neeche** = overvalued. Hover karo har point par exact numbers dekhne ke liye!

---
## 🏆 PART 10: Alpha & Performance Attribution

### Cell 10.1 — Jensen's Alpha Recap

$$ \alpha = R_{actual} - E(R_{CAPM}) $$

In [58]:
capm_df[['Ticker', 'Beta', "Jensen's Alpha (%)"]]

,Ticker,Beta,Jensen's Alpha (%)
4,NVDA,2.344,78.61
5,META,1.667,50.55
3,AMZN,1.540,19.73
2,GOOGL,1.314,14.22
1,MSFT,1.183,6.04
0,AAPL,1.034,14.64


### Cell 10.2 — Interactive Alpha Bar Chart (Color-Coded)

In [59]:
fig9 = px.bar(
    capm_df.sort_values("Jensen's Alpha (%)"),
    x="Jensen's Alpha (%)", y='Ticker', orientation='h',
    color="Jensen's Alpha (%)", color_continuous_scale='RdYlGn',
    title="🏆 Jensen's Alpha — Outperformance vs CAPM Prediction (Interactive)"
)
fig9.add_vline(x=0, line_color='black')
fig9.show()

---
## ⚖️ PART 11: Risk-Adjusted Performance (Sharpe Ratio)

### Cell 11.1 — Sharpe Ratio Formula

$$ \text{Sharpe Ratio} = \frac{R_{portfolio} - R_f}{\sigma_{portfolio}} $$

In [60]:
sharpe_results = []
for ticker in TICKERS + ['Market_SP500']:
    ret = annual_stats.loc[ticker, 'Annualized Return (%)'] / 100
    vol = annual_stats.loc[ticker, 'Annualized Volatility (%)'] / 100
    sharpe = (ret - RISK_FREE_RATE) / vol
    sharpe_results.append({'Ticker': ticker, 'Sharpe Ratio': round(sharpe, 3)})

sharpe_df = pd.DataFrame(sharpe_results).sort_values('Sharpe Ratio', ascending=False)
sharpe_df

,Ticker,Sharpe Ratio
4,NVDA,2.392
5,META,2.109
0,AAPL,1.557
3,AMZN,1.549
6,Market_SP500,1.398
2,GOOGL,1.296
1,MSFT,1.205


### Cell 11.2 — Interactive Sharpe Ratio Chart

In [61]:
fig10 = px.bar(
    sharpe_df, x='Sharpe Ratio', y='Ticker', orientation='h',
    color='Sharpe Ratio', color_continuous_scale='Blues',
    title="⚖️ Sharpe Ratio — Risk-Adjusted Performance (Interactive)"
)
fig10.show()

---
## 💼 PART 12: Portfolio-Level CAPM

### Cell 12.1 — Equal Weights Define Karna

In [62]:
weights = {ticker: 1/len(TICKERS) for ticker in TICKERS}
weights

{'AAPL': 0.16666666666666666,
 'MSFT': 0.16666666666666666,
 'GOOGL': 0.16666666666666666,
 'AMZN': 0.16666666666666666,
 'NVDA': 0.16666666666666666,
 'META': 0.16666666666666666}

### Cell 12.2 — Portfolio Beta (Weighted Average)

$$ \beta_{portfolio} = \sum_{i=1}^{n} w_i \beta_i $$

In [63]:
portfolio_beta = sum(weights[t] * beta_df.loc[beta_df['Ticker'] == t, 'Beta (Regression)'].values[0] for t in TICKERS)
print(f"Portfolio Beta: {portfolio_beta:.3f}")

Portfolio Beta: 1.514


### Cell 12.3 — Portfolio CAPM Expected Return

In [64]:
portfolio_capm_return = RISK_FREE_RATE + portfolio_beta * market_risk_premium
print(f"Portfolio CAPM Expected Return: {portfolio_capm_return*100:.2f}%")

Portfolio CAPM Expected Return: 31.73%


### Cell 12.4 — Portfolio Actual Historical Performance

In [65]:
portfolio_daily_returns = simple_returns[TICKERS].mean(axis=1)
portfolio_actual_annual = portfolio_daily_returns.mean() * TRADING_DAYS
portfolio_vol_annual = portfolio_daily_returns.std() * np.sqrt(TRADING_DAYS)
portfolio_sharpe = (portfolio_actual_annual - RISK_FREE_RATE) / portfolio_vol_annual

print(f"Actual Annualized Return:     {portfolio_actual_annual*100:.2f}%")
print(f"Actual Annualized Volatility: {portfolio_vol_annual*100:.2f}%")
print(f"Portfolio Sharpe Ratio:       {portfolio_sharpe:.3f}")
print(f"Jensen's Alpha:               {(portfolio_actual_annual - portfolio_capm_return)*100:.2f}%")

Actual Annualized Return:     62.36%
Actual Annualized Volatility: 23.98%
Portfolio Sharpe Ratio:       2.413
Jensen's Alpha:               30.63%


---
## 🗺️ PART 13: Final Risk-Return Dashboard

### Cell 13.1 — Combined Risk-Return Data Prepare Karna

In [66]:
dashboard_df = annual_stats.reset_index().rename(columns={'index': 'Ticker'})
dashboard_df = dashboard_df.merge(beta_df[['Ticker', 'Beta (Regression)']], on='Ticker', how='left')
dashboard_df

,Ticker,Annualized Return (%),Annualized Volatility (%),Beta (Regression)
0,NVDA,125.28,50.49,2.344
1,META,85.04,38.19,1.667
2,AMZN,51.93,30.62,1.540
3,GOOGL,42.36,29.21,1.314
4,AAPL,37.74,21.35,1.034
5,MSFT,31.82,22.68,1.183
6,Market_SP500,22.49,12.87,NaN


### Cell 13.2 — Interactive Bubble Chart (Risk vs Return vs Beta)

In [67]:
fig11 = px.scatter(
    dashboard_df, x='Annualized Volatility (%)', y='Annualized Return (%)',
    size=dashboard_df['Beta (Regression)'].fillna(1).abs() + 0.1,
    color='Ticker', text='Ticker',
    title="🗺️ Risk-Return Dashboard — Bubble Size = Beta (Interactive)",
    labels={'Annualized Volatility (%)': 'Risk (Volatility %)', 'Annualized Return (%)': 'Return (%)'}
)
fig11.update_traces(textposition='top center')
fig11.show()

C:\Users\Administrator\anaconda3\lib\site-packages\plotly\express\_core.py:1979: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



---
## 🎓 PART 14: Limitations, Conclusion & Further Study

### CAPM Ki Limitations:
1. **Single-Factor Model** — sirf market risk (Beta) consider karta hai. **Fama-French 3/5-Factor models** aur bhi factors add karte hain (size, value, profitability).
2. **Beta Instability** — rolling beta se dekha, time ke sath change hoti hai.
3. **Market Proxy Limitation** — S&P 500 sirf US large-cap; true "market portfolio" nahi.
4. **Efficient Market Assumption** — behavioral biases real duniya mein exist karte hain.
5. **Risk-Free Rate Assumption** — emerging markets (Pakistan) mein "risk-free" bhi fully risk-free nahi hota.
6. **Historical Data Limitation** — past beta, future ka guarantee nahi.

### Conclusion

CAPM finance ka ek **foundational framework** hai — risk aur return ke beech ka relationship samjhne ke liye. Real-world equity research, cost of capital, portfolio management mein widely use hota hai, lekin hamesha **judgment aur complementary models** ke sath.

### 📚 Further Reading:
- William Sharpe (1964) — *Capital Asset Prices: A Theory of Market Equilibrium* (Original CAPM paper)
- Eugene Fama & Kenneth French — *Three-Factor* aur *Five-Factor* models
- Zvi Bodie, Alex Kane, Alan Marcus — *Investments* (textbook)

### 🛠️ Skills Covered in This Notebook:
`pandas` (drop, inplace, loc/iloc, dropna, fillna) • `numpy` • `scipy.stats` • `statsmodels OLS` • `matplotlib`/`seaborn` • **`plotly.express` & `plotly.graph_objects`** interactive visualization • Financial modeling (CAPM, Beta, Alpha, Sharpe Ratio, Portfolio theory)

---
**Made with ❤️ — Teacher-Style World-Class CAPM Notebook**
